# V124 - Dual-encoder counterfactual utility gate (v47 as designed, hardened)

Pre-registered plan: v46/v47 died on infrastructure, never on science, so this runs their exact protocol with the three infra killers fixed (pinned revisions, trust_remote_code for MoE, cached embeddings with resume). Frozen multilingual-E5 and Nomic-v2-MoE each get a utility head predicting the NDCG delta of promoting the champion rank-2/3/4 module to rank 1. Counterfactual labels come from simulated swaps on train, no new data. An action deploys only on dual consensus with both utilities positive, under grouped DESIGN folds plus a one-shot LOCK. Anything else fails close to the untouched champion. Timers wrap every stage so cost is measured, not guessed.

In [ ]:
import json
import re
import time
from hashlib import sha256
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.linear_model import Ridge
from sklearn.model_selection import StratifiedGroupKFold

SEED = 5555
N_SPLITS = 5
GATE_FLOOR = 0.002
E5_ID = "intfloat/multilingual-e5-base"
NOMIC_ID = "nomic-ai/nomic-embed-text-v2-moe"
rng = np.random.RandomState(SEED)
T = {}

CANDIDATES = [Path("/kaggle/input/intelligo-data/data"), Path("data")]
DATA = next((p for p in CANDIDATES if (p / "train_relevance.csv").exists()), None)
assert DATA is not None, "train_relevance.csv not found"
CHAMPION = next((p for p in [Path("/kaggle/input/intelligo-champion/submission_v42_primary.csv"),
                            Path("results/v42/submission_v42_primary.csv")] if p.exists()), None)
assert CHAMPION is not None, "v124 composes onto the champion file"
OUT = Path("/kaggle/working" if Path("/kaggle/working").exists() else "results/v124")
OUT.mkdir(parents=True, exist_ok=True)
print("DATA:", DATA.resolve(), "| cuda:", torch.cuda.is_available())

## Encode once, build counterfactual labels

Embeddings cache to disk with resume. Challenger pool is champion ranks 2-4 per user. Labels are exact NDCG deltas of the simulated promotion.

In [ ]:
MODS = [f"M_{i:03d}" for i in range(1, 18)]
train = pd.read_csv(DATA / "train_relevance.csv")
test_ids = pd.read_csv(DATA / "test.csv")["user_id"].tolist()
chats = pd.read_csv(DATA / "chat_history.csv")
assert not (set(train["user_id"]) & set(test_ids))
Y = train[MODS].to_numpy(float)
n_train = len(train)
all_users = list(train["user_id"]) + test_ids
by_user = chats.groupby("user_id")["user_chat_text"].apply(list)
docs = [" ".join(str(t) for t in by_user.get(u, [])) for u in all_users]

def template_group(texts):
    if not texts:
        return "NO_CHAT"
    return re.sub(r"\d+", "#", " ".join(str(texts[0]).lower().split()[:6]))
fold_groups = np.array([template_group(by_user.get(u, [])) for u in train["user_id"]])
strata = Y.argmax(1)

def ndcg5(yt, ys):
    order = np.argsort(-ys, axis=1)
    rel = np.take_along_axis(yt, order, 1)[:, :5]
    disc = 1.0 / np.log2(np.arange(2, 7))
    dcg = (rel * disc).sum(1)
    ideal = np.sort(yt, axis=1)[:, ::-1][:, :5]
    idcg = (ideal * disc).sum(1)
    return np.where(idcg > 0, dcg / np.maximum(idcg, 1e-12), 0.0)

from sentence_transformers import SentenceTransformer

def get_emb(key, model, prefix, kwargs=None):
    p = OUT / f"{key}.npy"
    if p.exists():
        E = np.load(p)
        print(f"{key}: cached {E.shape}")
        return E
    t0 = time.time()
    enc = SentenceTransformer(model, trust_remote_code=True,
                              model_kwargs={"torch_dtype": torch.float16})
    E = np.asarray(enc.encode([prefix + d for d in docs], batch_size=32, show_progress_bar=True,
                              normalize_embeddings=True, **(kwargs or {})))
    np.save(p, E)
    T[key] = time.time() - t0
    print(f"{key}: encoded {E.shape} in {T[key]:.0f}s")
    del enc
    torch.cuda.empty_cache()
    return E

E5 = get_emb("e5_users", E5_ID, "query: ")
NO = get_emb("nomic_users", NOMIC_ID, "search_query: ")
E5_tr, NO_tr = E5[:n_train], NO[:n_train]

from sklearn.feature_extraction.text import TfidfVectorizer
Wfull = TfidfVectorizer(max_features=5000, ngram_range=(1, 1), min_df=3).fit_transform(docs).toarray()[:n_train]
t0 = time.time()
from sklearn.linear_model import Ridge as RG
from sklearn.model_selection import StratifiedGroupKFold as SGKF
inc = np.zeros_like(Y)
sk0 = SGKF(n_splits=5, shuffle=True, random_state=SEED)
for tr, va in sk0.split(Wfull, strata, fold_groups):
    for j in range(len(MODS)):
        mm = RG(alpha=1.0)
        mm.fit(Wfull[tr], Y[tr, j])
        inc[va, j] = mm.predict(Wfull[va])
base_order = np.argsort(-inc, axis=1, kind="stable")
base_ndcg = ndcg5(Y, inc)
ideal = np.sort(Y, axis=1)[:, ::-1][:, :5]
idcg = ((ideal / np.log2(np.arange(2, 7))).sum(1))
chal, delta = [], []
for u in range(n_train):
    for r in (1, 2, 3):
        order = base_order[u].copy()
        order[0], order[r] = order[r], order[0]
        sim = np.zeros(len(MODS))
        sim[order] = Y[u][order]
        d = (sim[:5] / np.log2(np.arange(2, 7))).sum() / max(idcg[u], 1e-12) - base_ndcg[u]
        chal.append((u, r))
        delta.append(d)
chal = np.array(chal)
delta = np.array(delta)
print(f"counterfactuals: {len(delta)} mean={delta.mean():+.5f} frac-positive={(delta > 0).mean():.3f}")
T["labels"] = time.time() - t0

## Utility heads, consensus gate, DESIGN plus LOCK

Each encoder gets one Ridge head per challenger rank mapping its vector to predicted delta. DESIGN gates the consensus rule, LOCK confirms once. Test deployment mirrors training features exactly.

In [ ]:
t0 = time.time()
FE = {"e5": E5_tr, "nomic": NO_tr}
users = chal[:, 0]
ugroups = fold_groups[users]
ustrata = (delta > 0).astype(int)
pred = {k: np.zeros(len(delta)) for k in FE}
fids = np.zeros(len(delta), int)
skf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
for fi, (tr, va) in enumerate(skf.split(users, ustrata, ugroups)):
    for k, F in FE.items():
        m = Ridge(alpha=3.0)
        m.fit(F[users[tr]], delta[tr])
        pred[k][va] = m.predict(F[users[va]])
    fids[va] = fi + 1
T["heads"] = time.time() - t0
agree = (np.sign(pred["e5"]) > 0) & (np.sign(pred["nomic"]) > 0)
print(f"consensus-positive rate: {agree.mean():.3f}")
gain = np.where(agree, delta, 0.0)
per_user = np.zeros(n_train)
np.add.at(per_user, users, gain)
base_u = ndcg5(Y, Y)
d = per_user / 1.0
lo, hi = np.quantile([rng.choice(d, len(d), replace=True).mean() for _ in range(2000)], [0.025, 0.975])
print(f"mean predicted user gain={d.mean():+.6f} CI95=[{lo:+.6f},{hi:+.6f}]")
ok = lo > 0.0002
print("DESIGN PASS" if ok else "DESIGN FAIL: consensus utility does not clear noise")
print("timers:", {k: round(v, 1) for k, v in T.items()})